# Preprocessing and Baseline
[Previous Notebook](dataset_audit.ipynb) | [**Next Notebook** (TBD)]()

Preprocessing the dataset is needed for this dataset as it include categorical features.

Here is the dataset from the last notebook

In [687]:
%store -r Teen_Mental_Health_Dataset
import pandas as pd

data = Teen_Mental_Health_Dataset.copy() # type: ignore
display(data.head())

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,not depressed
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,not depressed
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,not depressed
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,not depressed
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,not depressed


## Features and Target
To split the data into a **training set** and a **testing set**, I first need to split it into a **feature set** `X` and a **target set** `y`. 

I am not using a `LabelEncoder` here, instead I am manually setting the `depressed` values to `1.0` and `not depressed` to `0.0` so that `depressed` is the positive target result

In [688]:
target_labels = ["not depressed", "depressed"]

X = data.drop(columns=['depression_label'])
y = data['depression_label'].map(lambda x: target_labels.index(x))

## Training and Testing
Now that the data is split into a a **feature set** `X` and a **target set** `y`, I can split these into:
- **Training feature set** `X_train`
- **Testing feature set** `X_test`
- **Training target set** `y_train`
- **Testing target set** `y_test`

Using a `test_size=0.25`, i will split the data into 75% training and 25% testing. Since the dataset is so imbalanced, I chose a little bit higher percentage of training data than recommended in the slides to both mitigate overfitting and also to give the training set more opportunities to test positive target values.

I am using `stratify=y` to stratify `depression_label` so that the splits have a similar proportion of positive and negative target values

**N.B:** The data **MUST** be split before any machine learning practices are done to avoid leaking test information into the model.

In [689]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## Encoding Categorical features to numerical values.
I am using `OneHotEncoder` to encode the categorical features `gender` and `platform_usage`. This will create a new column for each unique value, i.e. `male`, `female`, `instagram`, etc. 

I am using `OrdinalEncoder` to encode the categorical features `social_interaction_level`. This will convert the values from `low`, `medium`, `high` to a range(0-2)

In [690]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

column_transformer = ColumnTransformer(
    transformers=[
        (
            "onehot_encoder",
            OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
            ['gender', 'platform_usage']
        ),
        (
            "ordinal_encoder",
            OrdinalEncoder(),
            ['social_interaction_level'],
        )
    ],
    remainder="passthrough"
)

# Baseline Model

For a baseline, i will use a simple `LogisticRegression` model. 

It is not computationally expensive and doesnt require much hyperparameter tuning. I can refer back to this baseline thoughout the remainder of the portfolio to see if using a more complex model is worth it, i.e. yields better results

In [691]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000, random_state=42)

## Pipeline

To keep preprocessing coherent and avoid leakage, I'll use the `sklearn.pipeline` API. 

I dont see any need for `StandardScaler` preprocessor (at least for this pipeline) as it seems as though all features are capped at a reasonable range and are non-negative.

In [692]:
from sklearn.pipeline import Pipeline, make_pipeline

preprocessor = Pipeline([
    ("column_transformer", column_transformer)
])

# store the preprocessing pipeline for later use
%store preprocessor

pipeline = make_pipeline(preprocessor, baseline_model)

Stored 'preprocessor' (Pipeline)


# Metrics

Before predicting and scoring, I'll outline the metrics I will and wont be looking at.

### **Will use**
- `Precision`: Out of all the items the model predicted as positive, how many were actually correct

- `Recall`: Out of all the actual positive items in the dataset, how many did the model manage to find

- `F1`: the harmonic mean of `precision` and `recall`

### **Wont use**
- `Accuracy:` as mentioned before, accuracy will not be a helpful metric for this study as over `97%` of the entries in the dataset are "not depressed", meaning I should expect at least `97%` accuracy no matter what model i use, which is misleading.

- `MAE`, `MSE`, `RMSE`, etc: Regression metrics do not apply

# Cross Validation

Im using `StratifiedKFold` to spilt the validation folds fairly.

Using `cross_validate` I can get the scores for all the metrics with one function call

In [693]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metrics = ["precision", "recall", "f1"]

cv = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring=metrics,
    return_train_score=True,
)

# format the results into a dataframe for easier analysis
df = pd.DataFrame(cv, index=[f"fold_{i+1}" for i in range(5)]).drop(columns=["fit_time", "score_time"])
train_df = df.drop(columns=[col for col in df.columns if col.startswith("test_")])
test_df = df.drop(columns=[col for col in df.columns if col.startswith("train_")])

def display_results(df, title):
    display(df.style.set_caption(title))
    results = df.describe().loc[["mean", "std", "min", "max"]]
    display(results)

display_results(train_df, "Training set performance")
print("=" * 80)
display_results(test_df, "Validation set performance")

,train_precision,train_recall,train_f1
fold_1,0.812500,0.684211,0.742857
fold_2,0.937500,0.789474,0.857143
fold_3,1.000000,0.777778,0.875000
fold_4,0.923077,0.666667,0.774194
fold_5,0.857143,0.666667,0.750000


,train_precision,train_recall,train_f1
mean,0.906044,0.716959,0.799839
std,0.072885,0.061417,0.061890
min,0.812500,0.666667,0.742857
max,1.000000,0.789474,0.875000


,test_precision,test_recall,test_f1
fold_1,0.500000,0.250000,0.333333
fold_2,0.500000,0.250000,0.333333
fold_3,0.750000,0.600000,0.666667
fold_4,0.600000,0.600000,0.600000
fold_5,1.000000,0.800000,0.888889


,test_precision,test_recall,test_f1
mean,0.67000,0.500000,0.564444
std,0.21095,0.242384,0.236539
min,0.50000,0.250000,0.333333
max,1.00000,0.800000,0.888889


## Interpretation of results

### Train
| Metric | Mean | Interpretation |
|-------|-------|-------|
| `Precision:` | `0.90` | Good precision for the training set |
| `Recall:` | `0.71` | Recall is ok |
| `F1:` | `0.79` | F1 is also pretty good as to be expected from `precision` and `recall` |

### Validation
| Metric | Mean | Interpretation |
|-------|-------|-------|
| `Precision:` | `0.67` | Precision is ok for the validation set |
| `Recall:` | `0.50` | Recall is pretty bad |
| `F1:` | `0.56` | F1 is also pretty bad due to the low recall |

As expected, the validation set yields worse results than the training set however the difference is not that concerning, so it is safe to say the baseline has minimal overfitting.

The standard deviation seems to be way higher on the validation set which indicated higher varience.

# Predictions

Now that I have cross validated my baseline model, I can fit the model to the training data and generate predictions using the held-out testing set. 

There should be no leakage as this is the first and only point that I will use the test set with the model.

In [694]:
from sklearn.metrics import check_scoring, classification_report, confusion_matrix

model = pipeline.fit(X_train, y_train)
predictions = model.predict(X_test)


## Confusion Matrix and Classification Report

### **Confusion Matrix**
Looking at the `confusion matrix` for the test results, we can see that the majority of predictions lie in the `TN` box, which is to be expected from a imbalanced dataset.

There seems to be 5 `FN` predictions an only 3 `TP` predictions (although 0 `FP` results!). This will certainly have an poor effect on the recall and f1 metrics of the test.

### **Classification Report**
Looking at the `classification report`, we can see a `precision` of `1.0`. this is because there were no `FP` predictions from this test, so all the predicted positive values were also correct.

The metric to really pay attention to is `recall` as it is only `0.38` which is quite bad. I should look to increase this metric with the following model experiments.

The `f1` score is also quite bad at `0.55`, but since the `precision` is so high already i should really focus on increasing the `recall`.


In [695]:
display(pd.DataFrame(confusion_matrix(y_test, predictions), index=target_labels, columns=target_labels).style.set_caption("Confusion Matrix"))

print("\nClassification report\n")
print(classification_report(y_test, predictions, zero_division=0, target_names=target_labels))

,not depressed,depressed
not depressed,292,0
depressed,5,3



Classification report

               precision    recall  f1-score   support

not depressed       0.98      1.00      0.99       292
    depressed       1.00      0.38      0.55         8

     accuracy                           0.98       300
    macro avg       0.99      0.69      0.77       300
 weighted avg       0.98      0.98      0.98       300



## Metrics

As already mentioned, here are the results of the desired metrics calculated using the `check_scoring` API.

In [697]:
results = {}
for metric in metrics:  
    results[metric] = check_scoring(model, scoring=metric)(model, X_test, y_test)

display(pd.DataFrame(results, index=["score"]).T)

,score
precision,1.000000
recall,0.375000
f1,0.545455


## [Go to the next notebook (TBD)]()